# Sonar Rock vs. Mine Classification

A binary classification experiment using 60 sonar measurements to distinguish mines (`M`) from rocks (`R`). The dataset is retrieved from the UCI Machine Learning Repository (ID 151).

This notebook covers data inspection, exploratory analysis, a DummyClassifier baseline, Logistic Regression, cross-validation, and held-out test evaluation.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 1
sns.set_theme(style='whitegrid')

## 2. Load the dataset

In [ ]:
sonar_dataset = fetch_ucirepo(id=151)
X = sonar_dataset.data.features.copy()
y = sonar_dataset.data.targets.iloc[:, 0].copy()
X.columns = [f'feature_{i + 1:02d}' for i in range(X.shape[1])]
y.name = 'target'
data = X.copy()
data['target'] = y
data.head()

## 3. Initial data inspection

In [ ]:
print('Dataset shape:', data.shape)
display(data.dtypes.value_counts())
display(X.describe().T)
display(y.value_counts().rename('count').to_frame())
print('Missing values:', int(data.isna().sum().sum()))
print('Duplicate rows:', int(data.duplicated().sum()))

## 4. Exploratory data analysis

In [ ]:
X.hist(figsize=(18, 20), bins=20, layout=(10, 6))
plt.suptitle('Distributions of sonar features', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
feature_to_plot = X.columns[0]
plt.figure(figsize=(8, 5))
for label, class_name in [('M', 'Mine'), ('R', 'Rock')]:
    plt.hist(X.loc[y == label, feature_to_plot], bins=15, alpha=0.55, label=class_name)
plt.xlabel(feature_to_plot)
plt.ylabel('Frequency')
plt.title(f'{feature_to_plot}: Mine vs. Rock')
plt.legend()
plt.show()

## 5. Train/test split

Stratification preserves approximate class proportions. The test set is held out for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Training set:', X_train.shape)
print('Test set:', X_test.shape)

## 6. Baseline model

The most-frequent DummyClassifier always predicts the majority class from the training set. It provides a simple reference score.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f'Baseline accuracy: {baseline_accuracy:.4f}')

## 7. Logistic Regression pipeline

StandardScaler is fitted inside the pipeline, so scaling is learned from training folds only during cross-validation.

In [ ]:
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)

## 8. Cross-validation on the training set

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
)
cv_summary = pd.DataFrame({
    metric: cv_results[f'test_{metric}']
    for metric in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
})
display(cv_summary)
print('Mean CV accuracy:', round(cv_summary['accuracy'].mean(), 4))
print('Mean CV macro F1:', round(cv_summary['f1_macro'].mean(), 4))

## 9. Final evaluation on the held-out test set

In [ ]:
model.fit(X_train, y_train)
test_predictions = model.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 4))
print(classification_report(y_test, test_predictions, labels=['M', 'R'], target_names=['Mine', 'Rock'], zero_division=0))
cm = confusion_matrix(y_test, test_predictions, labels=['M', 'R'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Mine', 'Rock'])
disp.plot(values_format='d')
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

## 10. Findings and limitations

The dataset is small (208 observations), so performance estimates can vary with the split. Cross-validation and test-set results should be considered together. This is an educational experiment, not a validated operational mine-detection system. Further work could compare SVM, Random Forest, and k-NN, and use repeated stratified cross-validation.